# NeoBank India — Customer Churn Analysis
## Part 1: Data Extraction & Cleaning

---

**Project Overview**

NeoBank India is a mid-sized private bank operating across Metro, Urban, Semi-Urban, and Rural markets. Despite steady growth in customer acquisition, the retention team has flagged a significant rise in churned customers over the past year. This notebook covers extracting raw data from the bank's normalised database, identifying and resolving data quality issues, and producing a clean dataset ready for analysis.

**Workflow:** Extract → Inspect → Clean → Export

> **Note on Data:** The SQLite database (`neobank_churn.db`) used in this project is synthetically generated to simulate realistic banking data quality issues — including inconsistent encodings, mixed formats, outliers, and missing values. All customer records are fictional and created for portfolio/educational purposes only.

---

**Database Structure**

Data is stored across 6 normalised tables in a SQLite database:

| Table | Description |
|---|---|
| `customers` | Age, gender, city tier, occupation, tenure |
| `accounts` | Account type, average balance, credit score |
| `products` | Banking products held per customer |
| `behaviour` | Transactions, app logins, digital engagement |
| `service` | Support calls and late payment history |
| `churn` | Target variable |

---

**Tech Stack:** Python · pandas · NumPy · Matplotlib · Seaborn · SQLite

---

---
## 1. Import Libraries

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

---
## 2. Connect to Database & Verify Tables

In [ ]:
conn = sqlite3.connect('neobank_churn.db')

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("Tables in database:")
print(tables)

---
## 3. Extract Data — SQL JOIN Across All 6 Tables

All 6 tables are joined on `customer_id` to produce a single flat dataset for analysis.

In [ ]:
query = """
    SELECT
        c.customer_id, c.age, c.gender, c.city_tier,
        c.occupation, c.tenure_months,
        a.account_type, a.avg_balance, a.credit_score,
        p.has_credit_card, p.has_personal_loan, p.has_home_loan,
        p.has_fixed_deposit, p.has_demat_account, p.has_insurance,
        p.num_products,
        b.monthly_transactions, b.days_since_last_transaction,
        b.app_logins_per_month, b.failed_transactions,
        b.digital_engagement,
        s.support_calls, s.late_payments,
        ch.churn
    FROM customers c
    LEFT JOIN accounts  a  ON c.customer_id = a.customer_id
    LEFT JOIN products  p  ON c.customer_id = p.customer_id
    LEFT JOIN behaviour b  ON c.customer_id = b.customer_id
    LEFT JOIN service   s  ON c.customer_id = s.customer_id
    LEFT JOIN churn     ch ON c.customer_id = ch.customer_id
"""

df = pd.read_sql(query, conn)
conn.close()

print(f"Shape: {df.shape}")
df.head()

---
## 4. Initial Inspection

Before making any changes, we inspect the dataset to understand its structure, data types, and missing values.

In [ ]:
df.info()

In [ ]:
# Missing value summary
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# Visual check — which columns have missing values and do they overlap
plt.figure(figsize=(10, 5))
sns.heatmap(df[['credit_score', 'avg_balance']].isnull(), cbar=False)
plt.title('Missing Values Pattern — credit_score and avg_balance')
plt.show()

---
## 5. Handling Duplicate Records

**Observation:** The dataset contains duplicate `customer_id` entries. On inspection, some duplicates are fully identical rows while others have the same customer ID but differ only in `support_calls` — suggesting the same customer record was captured more than once with a minor difference.

**Decision:** For fully identical rows, we drop duplicates. For partial duplicates where only `support_calls` differs, we aggregate by summing `support_calls` and taking the first value for all other columns — since support calls are additive events.

In [ ]:
print(f"Total rows before deduplication      : {len(df):,}")
print(f"Duplicate customer_id entries        : {df['customer_id'].duplicated().sum()}")

# Inspect duplicate rows
df[df['customer_id'].duplicated(keep=False)].sort_values('customer_id').head(10)

In [ ]:
# For partial duplicates: sum support_calls, take first for all other columns
agg_dict = {}
for col in df.columns:
    if col == 'support_calls':
        agg_dict[col] = 'sum'
    elif col != 'customer_id':
        agg_dict[col] = 'first'

df = df.groupby('customer_id', as_index=False).agg(agg_dict)

print(f"Total rows after deduplication       : {len(df):,}")
print(f"Remaining duplicate customer_ids     : {df['customer_id'].duplicated().sum()}")

---
## 6. Standardising Categorical Columns

Several columns contain the same values stored in different formats — mixed case, abbreviations, and whitespace. Each is inspected and standardised below.

### 6.1 Gender
**Observed:** Column contains `Male`, `Female`, `M`, `F`, `MALE` — all representing two distinct values.

In [ ]:
print("Unique values before cleaning:")
print(df['gender'].unique())

df['gender'] = df['gender'].str.strip()
df['gender'] = df['gender'].replace({
    'M'     : 'Male',
    'MALE'  : 'Male',
    'F'     : 'Female',
    'FEMALE': 'Female'
})

print("\nUnique values after cleaning:")
print(df['gender'].unique())

### 6.2 City Tier
**Observed:** Same tier is stored as `Metro`, `metro`, `METRO`, `Tier 1`, `tier1`, `Tier-1` — and some values have leading/trailing whitespace.

In [ ]:
print("Unique values before cleaning:")
print(sorted(df['city_tier'].str.strip().unique()))

df['city_tier'] = df['city_tier'].str.strip().str.lower()
df['city_tier'] = df['city_tier'].replace({
    'metro'      : 'Metro',
    'tier 1'     : 'Metro',
    'tier1'      : 'Metro',
    'tier-1'     : 'Metro',
    'urban'      : 'Urban',
    'tier 2'     : 'Urban',
    'tier2'      : 'Urban',
    'semi-urban' : 'Semi-Urban',
    'semi urban' : 'Semi-Urban',
    'semiurban'  : 'Semi-Urban',
    'semi_urban' : 'Semi-Urban',
    'tier 3'     : 'Semi-Urban',
    'rural'      : 'Rural',
    'tier 4'     : 'Rural',
    'tier4'      : 'Rural'
})

print("\nUnique values after cleaning:")
print(df['city_tier'].value_counts())

### 6.3 Occupation
**Observed:** Values include `SELF-EMPLOYED`, `self-employed`, `Self Employed`, `employed`, `Business` — all referring to the same categories.

In [ ]:
print("Unique values before cleaning:")
print(sorted(df['occupation'].str.strip().unique()))

df['occupation'] = df['occupation'].str.strip().str.lower()
df['occupation'] = df['occupation'].replace({
    'salaried'     : 'Salaried',
    'employed'     : 'Salaried',
    'self-employed': 'Self-Employed',
    'self employed': 'Self-Employed',
    'business'     : 'Self-Employed',
    'student'      : 'Student',
    'stud'         : 'Student',
    'retired'      : 'Retired',
    'ret'          : 'Retired'
})

print("\nUnique values after cleaning:")
print(df['occupation'].value_counts())

### 6.4 Account Type
**Observed:** Values include full names (`Savings Account`), partial names (`Savings`), and short codes (`SA`, `CA`, `FD`). Whitespace also present.

In [ ]:
print("Unique values before cleaning:")
print(sorted(df['account_type'].str.strip().unique()))

df['account_type'] = df['account_type'].str.strip().str.lower()
df['account_type'] = df['account_type'].replace({
    'savings account'       : 'Savings Account',
    'savings'               : 'Savings Account',
    'sa'                    : 'Savings Account',
    'salary account'        : 'Salary Account',
    'salary'                : 'Salary Account',
    'sal'                   : 'Salary Account',
    'current account'       : 'Current Account',
    'current'               : 'Current Account',
    'ca'                    : 'Current Account',
    'fixed deposit account' : 'Fixed Deposit Account',
    'fixed deposit'         : 'Fixed Deposit Account',
    'fd account'            : 'Fixed Deposit Account',
    'fd'                    : 'Fixed Deposit Account'
})

print("\nUnique values after cleaning:")
print(df['account_type'].value_counts())

### 6.5 Digital Engagement
**Observed:** Same three levels stored as `High`, `HIGH`, `high`, `H`, `Med`, `LOW`, `L` etc.

In [ ]:
print("Unique values before cleaning:")
print(df['digital_engagement'].unique())

df['digital_engagement'] = df['digital_engagement'].str.strip().str.lower()
df['digital_engagement'] = df['digital_engagement'].replace({
    'high'   : 'High',
    'h'      : 'High',
    'medium' : 'Medium',
    'med'    : 'Medium',
    'low'    : 'Low',
    'l'      : 'Low'
})

print("\nUnique values after cleaning:")
print(df['digital_engagement'].value_counts())

### 6.6 Churn Column
**Observed:** The churn column contains 8 different representations of the same two outcomes — `Yes`, `No`, `Y`, `N`, `yes`, `no`, `1`, `0`. This indicates the column was populated inconsistently across records.

**Decision:** Standardise to binary integer — `1` = churned, `0` = not churned.

In [ ]:
print("Unique values before cleaning:")
print(df['churn'].unique())

df['churn'] = df['churn'].replace({
    'Yes' : 1, 'yes' : 1, 'Y' : 1, '1' : 1,
    'No'  : 0, 'no'  : 0, 'N' : 0, '0' : 0
})
df['churn'] = df['churn'].astype(int)

print("\nUnique values after cleaning:")
print(df['churn'].unique())
print(f"\nChurn rate: {df['churn'].mean()*100:.1f}%")

---
## 7. Cleaning Numeric Columns

### 7.1 Average Balance — Remove Currency Symbol
**Observed:** Some values in `avg_balance` are stored as strings with a rupee symbol and comma separator (e.g., `₹12,500`). These need to be converted to numeric before any analysis can be done.

In [ ]:
# Check current dtype and sample values
print(f"Current dtype: {df['avg_balance'].dtype}")
print("\nSample values with currency symbol:")
mask = df['avg_balance'].astype(str).str.contains('₹', na=False)
print(f"Count: {mask.sum()}")
print(df.loc[mask, 'avg_balance'].head())

# Strip symbol and convert
df['avg_balance'] = (
    df['avg_balance']
    .astype(str)
    .str.replace('₹', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)
df['avg_balance'] = pd.to_numeric(df['avg_balance'], errors='coerce')

print(f"\nDtype after cleaning: {df['avg_balance'].dtype}")

### 7.2 Days Since Last Transaction — Remove Text Units
**Observed:** Column contains both plain integers (e.g., `45`) and string formats with units (e.g., `45 days`, `45d`). These need to be standardised to integer.

In [ ]:
# Check mixed formats
mask = df['days_since_last_transaction'].astype(str).str.contains('[a-zA-Z]', na=False)
print(f"Rows with text units: {mask.sum()}")
print(df.loc[mask, 'days_since_last_transaction'].head(5).values)

# Extract numeric part only
df['days_since_last_transaction'] = (
    df['days_since_last_transaction']
    .astype(str)
    .str.extract(r'(\d+)')[0]
)
df['days_since_last_transaction'] = pd.to_numeric(df['days_since_last_transaction'], errors='coerce').astype(int)

print(f"\nDtype after cleaning: {df['days_since_last_transaction'].dtype}")

---
## 8. Handling Outliers

### 8.1 Age — Invalid Values
**Observed:** Some records have `age = 0` which is not a valid customer age. These records have valid data in all other columns, so the issue is specific to the age field.

**Decision:** Replace 0 with NaN, then impute using the median age for each occupation group — since age distribution varies meaningfully across occupations (students are younger, retired customers are older).

In [ ]:
print(f"Age range in raw data: {df['age'].min()} to {df['age'].max()}")
print(f"Rows with age = 0    : {(df['age'] == 0).sum()}")

# Median age by occupation — used for imputation reference
print("\nMedian age by occupation:")
print(df.groupby('occupation')['age'].median())

# Replace 0 with NaN and impute
df['age'] = df['age'].replace(0, np.nan)
df['age'] = df.groupby('occupation')['age'].transform(lambda x: x.fillna(x.median()))
df['age'] = df['age'].astype(int)

print(f"\nAge range after cleaning: {df['age'].min()} to {df['age'].max()}")
print(f"Remaining missing       : {df['age'].isnull().sum()}")

### 8.2 Average Balance — Outlier Detection
**Observed:** After converting `avg_balance` to numeric, we check the distribution to identify extreme values that may not be genuine.

We use three visualisations — a histogram for overall distribution, a KDE plot for the density shape, and a boxplot to identify outliers.

In [ ]:
# Distribution before handling outliers
plt.figure(figsize=(8, 5))
plt.hist(df['avg_balance'].dropna(), bins=30)
plt.xlabel('Average Balance (₹)')
plt.ylabel('Frequency')
plt.title('Distribution of Average Balance — Before Cleaning')
plt.show()

In [ ]:
# KDE plot to see density shape
plt.figure(figsize=(8, 5))
sns.kdeplot(df['avg_balance'].dropna(), fill=True)
plt.title('Average Balance — Density Plot')
plt.xlabel('Average Balance (₹)')
plt.show()

In [ ]:
# Boxplot to clearly identify outlier position
plt.figure(figsize=(10, 2))
sns.boxplot(x=df['avg_balance'].dropna())
plt.title('Average Balance — Boxplot (Outlier Detection)')
plt.show()

In [ ]:
# Statistical summary to confirm the outlier
print(df['avg_balance'].describe())

# Inspect the maximum value
print(f"\nMax value in avg_balance:")
print(df[df['avg_balance'] == df['avg_balance'].max()][['customer_id', 'avg_balance', 'occupation', 'city_tier']])

**Finding:** The maximum value is `99,999,999` — which is clearly not a realistic bank balance given the distribution of all other values (which range up to ~₹5-6 lakhs). This appears to be a data entry error. 

**Decision:** Replace this value with NaN and impute using the group median.

In [ ]:
# Remove the extreme outlier
df['avg_balance'] = df['avg_balance'].replace(99999999, np.nan)

# Verify it's gone
print(f"Max value after removal: {df['avg_balance'].max():,.0f}")

In [ ]:
# KDE plot after removing outlier — distribution is now clean
plt.figure(figsize=(8, 5))
sns.kdeplot(df['avg_balance'].dropna(), fill=True)
plt.title('Average Balance — Density Plot After Cleaning')
plt.xlabel('Average Balance (₹)')
plt.show()

### 8.3 Credit Score — Invalid Values
**Observed:** Credit scores should fall between 300 and 850 (standard bureau range). Values outside this range are not valid scores.

**Decision:** Replace out-of-range values with NaN and impute using group median.

In [ ]:
print(f"Credit score range: {df['credit_score'].min()} to {df['credit_score'].max()}")
print(f"Out-of-range values (outside 300-850): {((df['credit_score'] < 300) | (df['credit_score'] > 850)).sum()}")

# Sample of out-of-range records
print("\nSample out-of-range records:")
print(df[(df['credit_score'] < 300) | (df['credit_score'] > 850)][['customer_id', 'credit_score']].head())

In [ ]:
# Replace invalid values with NaN
df.loc[(df['credit_score'] < 300) | (df['credit_score'] > 850), 'credit_score'] = np.nan

print(f"Missing credit scores after flagging: {df['credit_score'].isnull().sum()}")

---
## 9. Handling Missing Values

### 9.1 Credit Score
Before imputing, we check whether missing values are randomly distributed or concentrated in specific segments — since the imputation strategy should match the pattern.

In [ ]:
print(f"Missing credit scores: {df['credit_score'].isnull().sum()} ({df['credit_score'].isnull().mean()*100:.1f}%)")

# Check correlation with other numeric columns before imputing
print("\nCorrelation of credit_score with other numeric columns:")
print(df.corr(numeric_only=True)['credit_score'].sort_values(ascending=False))

In [ ]:
# Check if missing values cluster in specific segments
df['cs_missing'] = df['credit_score'].isnull().astype(int)

print("Missing rate by city tier:")
print(df.groupby('city_tier')['cs_missing'].mean().apply(lambda x: f"{x*100:.1f}%"))

print("\nMissing rate for tenure <= 3 months:")
print(df[df['tenure_months'] <= 3].groupby('tenure_months')['cs_missing'].mean().apply(lambda x: f"{x*100:.1f}%"))

df.drop(columns=['cs_missing'], inplace=True)

**Finding:** Missing values are not random — they are concentrated in newer customers (short tenure) and Rural/Semi-Urban segments. This means a global median would be inappropriate. We impute using the group median segmented by occupation, account type, and city tier to preserve segment-level differences.

In [ ]:
df['credit_score'] = df.groupby(['occupation', 'account_type', 'city_tier'])['credit_score'].transform(
    lambda x: x.fillna(x.median())
)

print(f"Missing credit scores after imputation: {df['credit_score'].isnull().sum()}")
print("\nCredit score summary after cleaning:")
print(df['credit_score'].describe())

### 9.2 Average Balance
We check the correlation and group medians before deciding on the imputation strategy.

In [ ]:
print(f"Missing avg_balance: {df['avg_balance'].isnull().sum()} ({df['avg_balance'].isnull().mean()*100:.1f}%)")

# Correlation check
print("\nCorrelation of avg_balance with other numeric columns:")
print(df.corr(numeric_only=True)['avg_balance'].sort_values(ascending=False))

In [ ]:
# Group medians — to determine best segmentation for imputation
print("Median avg_balance by occupation:")
print(df.groupby('occupation')['avg_balance'].median())

print("\nMedian avg_balance by account type:")
print(df.groupby('account_type')['avg_balance'].median())

print("\nMedian avg_balance by city tier:")
print(df.groupby('city_tier')['avg_balance'].median())

**Finding:** Average balance varies significantly across occupation, account type, and city tier. Imputing with group median across all three segments gives a more accurate estimate than using a single global median.

In [ ]:
df['avg_balance'] = df.groupby(['occupation', 'account_type', 'city_tier'])['avg_balance'].transform(
    lambda x: x.fillna(x.median())
)

print(f"Missing avg_balance after imputation: {df['avg_balance'].isnull().sum()}")
print("\nAvg balance summary after cleaning:")
print(df['avg_balance'].describe())

---
## 10. Final Column Ordering & Data Type Check

In [ ]:
# Move churn to last column
cols = [col for col in df.columns if col != 'churn']
df = df[cols + ['churn']]

# Verify all data types
print(df.dtypes)

---
## 11. Final Validation

In [ ]:
print("=" * 50)
print("FINAL DATASET SUMMARY")
print("=" * 50)
print(f"Shape              : {df.shape}")
print(f"Duplicate rows     : {df.duplicated().sum()}")
print(f"Missing values     : {df.isnull().sum().sum()}")
print(f"Churn rate         : {df['churn'].mean()*100:.1f}%")
print(f"Age range          : {df['age'].min()} to {df['age'].max()}")
print(f"Tenure range       : {df['tenure_months'].min()} to {df['tenure_months'].max()} months")
print(f"Credit score range : {df['credit_score'].min():.0f} to {df['credit_score'].max():.0f}")
print(f"Avg balance range  : ₹{df['avg_balance'].min():,.0f} to ₹{df['avg_balance'].max():,.0f}")

In [ ]:
df_final = df.copy()
df_final.head(10)

---
## 12. Export Clean Dataset

In [ ]:
df_final.to_csv('neobank_cleaned.csv', index=False)
print("Clean dataset saved: neobank_cleaned.csv")
print(f"Rows    : {len(df_final):,}")
print(f"Columns : {df_final.shape[1]}")

---
## Cleaning Summary

| Issue | Column(s) | What Was Observed | Action Taken |
|---|---|---|---|
| Duplicate rows | All | 55 duplicate customer IDs — some exact, some differing only in support_calls | Removed exact duplicates; aggregated support_calls for partial duplicates |
| Inconsistent values | `gender` | M, F, MALE alongside Male, Female | Standardised to Male / Female |
| Inconsistent values | `city_tier` | 12 different representations of 4 tiers | Mapped all variants to Metro / Urban / Semi-Urban / Rural |
| Inconsistent values | `occupation` | Mixed case, abbreviations, alternate terms | Standardised to 4 categories |
| Inconsistent values | `account_type` | Short codes (SA, CA, FD) mixed with full names | Mapped all to full standard names |
| Inconsistent values | `digital_engagement` | H, Med, L mixed with full words | Standardised to High / Medium / Low |
| Inconsistent values | `churn` | 8 different representations of Yes/No | Standardised to 0 / 1 |
| Non-numeric format | `avg_balance` | Currency symbol ₹ and commas embedded in values | Stripped symbol, converted to float |
| Non-numeric format | `days_since_last_transaction` | Mixed integer and string formats (45 vs '45 days') | Extracted numeric part |
| Invalid value | `age` | Age = 0 present in dataset | Replaced with occupation-group median |
| Extreme outlier | `avg_balance` | Value of 99,999,999 — inconsistent with all other values | Replaced with NaN and imputed with group median |
| Out-of-range values | `credit_score` | Values above 850 (valid bureau range is 300–850) | Replaced with NaN and imputed with group median |
| Missing values | `credit_score`, `avg_balance` | Concentrated in newer and Rural/Semi-Urban customers | Imputed with segment-level group median |

---
*Next: `02_EDA.ipynb` — Exploratory Data Analysis*